# GTSRB Research: Experiment 2 — AugMix ResNet-18 Robustness Benchmark

This notebook implements **Experiment 2 (Data-Centric Robustness)** from our controlled research ladder:
$$X \to \text{AugMix} \to \text{ResNet-18} \to \text{Jensen-Shannon Consistency Loss } (\lambda_{\text{JSD}} = 12)$$

### Methodological Controls (Strictly Identical to Baseline):
1. **Architecture:** Pretrained ResNet-18 with Kaiming Normal 43-class head and Zero-$\gamma$ BN init.
2. **Optimization:** SGD (Nesterov $\mu=0.9$, lr=0.05, weight decay $10^{-4}$ with No-Bias-Decay rule).
3. **Schedule:** 3-epoch linear warmup + half-period Cosine Annealing decay over 25 epochs.
4. **Domain Constraints:** Rotation $\le \pm 10^\circ$, Shear $\le \pm 8^\circ$, Translation $\le \pm 4$ px, strictly zero test-corruption leakage.
5. **Evaluation:** Clean accuracy, Macro-F1, ECE, and 15 held-out corruptions $\times$ 5 severities ($mCA$ and $RPD$).

In [ ]:
# --- Step 1: Environment Setup & GPU Verification ---
import os
from google.colab import drive
drive.mount('/content/drive')

# Check if dataset is already unzipped from baseline run
if not os.path.exists('/content/ML_Project/german'):
    print('Extracting base dataset from Drive...')
    !cp "/content/drive/MyDrive/ML_Project_Data.zip" /content/
    !unzip -q /content/ML_Project_Data.zip -d /content/ML_Project
else:
    print('Base dataset already present at /content/ML_Project!')

# Update Exp 2 code if uploaded as tiny zip
if os.path.exists('/content/drive/MyDrive/exp2_augmix_code.zip'):
    print('Updating Exp 2 code from exp2_augmix_code.zip...')
    !unzip -o -q "/content/drive/MyDrive/exp2_augmix_code.zip" -d /content/ML_Project

%cd /content/ML_Project

import torch
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device     : {torch.cuda.get_device_name(0)}")
!nvidia-smi

In [ ]:
# --- Step 2: AugMix Visual Inspection (Clean vs Aug1 vs Aug2) ---
import matplotlib.pyplot as plt
import numpy as np
from src.augmentations.augmix import AugMixDataset
from src.dataset import IMAGENET_MEAN, IMAGENET_STD

train_ds = AugMixDataset('data/master_v1/metadata/train.csv', 'german', target_size=64, is_training=True)
print(f"AugMix Train samples: {len(train_ds):,}")

# Plot 4 sample signs showing Clean, AugView 1, and AugView 2 side-by-side
fig, axes = plt.subplots(4, 3, figsize=(12, 12))
col_titles = ['Clean View', 'AugMix View 1', 'AugMix View 2']

for row_idx, sample_idx in enumerate([100, 500, 1200, 2500]):
    item = train_ds[sample_idx]
    for col_idx, key in enumerate(['image_clean', 'image_aug1', 'image_aug2']):
        ax = axes[row_idx, col_idx]
        img_np = item[key].cpu().numpy().transpose(1, 2, 0)
        img_np = (img_np * IMAGENET_STD + IMAGENET_MEAN).clip(0, 1)
        ax.imshow(img_np)
        if row_idx == 0:
            ax.set_title(col_titles[col_idx], fontsize=12, fontweight='bold')
        if col_idx == 0:
            ax.set_ylabel(item['class_name'], fontsize=9)
        ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# --- Step 3: Train AugMix ResNet-18 (Colab T4 GPU) ---
# 25 epochs with 3x concatenated forward passes takes ~8-10 minutes on T4 with AMP
!python train_augmix.py \
    --data-root german \
    --meta-dir data/master_v1/metadata \
    --output-dir checkpoints/augmix_resnet18 \
    --seed 42 \
    --epochs 25 \
    --batch-size 128 \
    --lr 0.05 \
    --lambda-jsd 12.0 \
    --severity 3 \
    --width 3 \
    

In [ ]:
# --- Step 4: Plot Loss & Validation Curves ---
import json
import pandas as pd

with open('checkpoints/augmix_resnet18/training_history.json', 'r') as f:
    history = json.load(f)

epochs_df = pd.DataFrame(history['epochs'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss decomposition (Total, CE, JSD)
ax1.plot(epochs_df['epoch'], epochs_df['train_loss'], label='Total Train Loss', color='blue', lw=2)
ax1.plot(epochs_df['epoch'], epochs_df['train_ce'], label='Clean CE Loss', color='cyan', linestyle='--')
ax1.plot(epochs_df['epoch'], epochs_df['train_jsd'], label='JSD Consistency', color='orange', linestyle=':')
ax1.plot(epochs_df['epoch'], epochs_df['val_loss'], label='Val Loss', color='red', lw=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss Convergence (Cross-Entropy + JSD)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy & Macro-F1
ax2.plot(epochs_df['epoch'], epochs_df['val_acc'] * 100, label='Val Accuracy (%)', color='green', lw=2)
ax2.plot(epochs_df['epoch'], epochs_df['val_macro_f1'] * 100, label='Val Macro-F1 (%)', color='purple', lw=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Score (%)')
ax2.set_title('Validation Performance')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- Step 5: Clean Test & Robustness Benchmark (15 Corruptions x 5 Severities) ---
!python evaluate_robustness.py \
    --checkpoint checkpoints/augmix_resnet18/best_augmix_resnet18.pt \
    --data-root german \
    --meta-path data/master_v1/metadata/test.csv \
    --output-dir reports/augmix_resnet18 \
    --batch-size 128

In [ ]:
# --- Step 6: Comparative Analysis: Baseline vs. AugMix ---
import json
import pandas as pd
from pathlib import Path

base_summary_path = Path('reports/baseline_resnet18/evaluation_summary.json')
aug_summary_path = Path('reports/augmix_resnet18/evaluation_summary.json')

if base_summary_path.exists() and aug_summary_path.exists():
    with open(base_summary_path) as f:
        base_res = json.load(f)
    with open(aug_summary_path) as f:
        aug_res = json.load(f)
        
    comparison = pd.DataFrame([
        {
            'Metric': 'Clean Test Accuracy',
            'Baseline': f"{base_res['clean_accuracy']*100:.2f}%",
            'AugMix (Exp 2)': f"{aug_res['clean_accuracy']*100:.2f}%",
            'Delta': f"{(aug_res['clean_accuracy'] - base_res['clean_accuracy'])*100:+.2f}%",
        },
        {
            'Metric': 'Clean Test Macro-F1',
            'Baseline': f"{base_res['clean_macro_f1']*100:.2f}%",
            'AugMix (Exp 2)': f"{aug_res['clean_macro_f1']*100:.2f}%",
            'Delta': f"{(aug_res['clean_macro_f1'] - base_res['clean_macro_f1'])*100:+.2f}%",
        },
        {
            'Metric': 'Expected Calibration Error (ECE)',
            'Baseline': f"{base_res['clean_ece']:.4f}",
            'AugMix (Exp 2)': f"{aug_res['clean_ece']:.4f}",
            'Delta': f"{(aug_res['clean_ece'] - base_res['clean_ece']):+.4f}",
        },
        {
            'Metric': 'Mean Corruption Accuracy (mCA)',
            'Baseline': f"{base_res['mean_corruption_accuracy']*100:.2f}%",
            'AugMix (Exp 2)': f"{aug_res['mean_corruption_accuracy']*100:.2f}%",
            'Delta': f"{(aug_res['mean_corruption_accuracy'] - base_res['mean_corruption_accuracy'])*100:+.2f}%",
        },
        {
            'Metric': 'Relative Perf Drop (RPD)',
            'Baseline': f"{base_res['relative_performance_drop']*100:.2f}%",
            'AugMix (Exp 2)': f"{aug_res['relative_performance_drop']*100:.2f}%",
            'Delta': f"{(aug_res['relative_performance_drop'] - base_res['relative_performance_drop'])*100:+.2f}%",
        },
    ])
    display(comparison)
else:
    print("Baseline or AugMix summary JSON not found. Displaying AugMix results only:")
    matrix_df = pd.read_csv('reports/augmix_resnet18/robustness_matrix.csv')
    display(matrix_df)

# Copy checkpoints and reports to Google Drive
!mkdir -p "/content/drive/MyDrive/ML_Project_Results/augmix_resnet18"
!cp -r checkpoints/augmix_resnet18/* "/content/drive/MyDrive/ML_Project_Results/augmix_resnet18/"
!cp -r reports/augmix_resnet18/* "/content/drive/MyDrive/ML_Project_Results/augmix_resnet18/"
print("\nAll AugMix results backed up to Google Drive successfully!")